In [ ]:
import pandas as pd
import altair as alt

alt.data_transformers.enable("default", max_rows = None)

datasetUrl = "https://raw.githubusercontent.com/UIUC-iSchool-DataViz/is445_data/main/building_inventory.csv"
datasetUrl

In [ ]:
buildingDatasetRaw = pd.read_csv(datasetUrl)
buildingDatasetRaw

In [ ]:
buildingDatasetRaw.columns

In [ ]:
buildingDatasetRaw.isna().sum()

In [ ]:
numericFeatures = buildingDatasetRaw.select_dtypes(include = ["int64", "float64"]).columns 
numericFeatures

In [ ]:
categoricalFeatures = buildingDatasetRaw.select_dtypes(include = ["object", "category"]).columns 
categoricalFeatures

In [ ]:
buildingDatasetRaw["Agency Name"].unique()

In [ ]:
agencyCategories = {
    "University of Illinois": "Higher Education",
    "Southern Illinois University": "Higher Education",
    "Illinois State University": "Higher Education",
    "Northern Illinois University": "Higher Education",
    "Eastern Illinois University": "Higher Education",
    "Western Illinois University": "Higher Education",
    "Northeastern Illinois University": "Higher Education",
    "Chicago State University": "Higher Education",
    "Governors State University": "Higher Education",
    "Illinois Community College Board": "Education Administration",
    "Illinois Board of Higher Education": "Education Administration",
    "IL State Board of Education": "Education Administration",
    "Department of Corrections": "Corrections & Justice",
    "Department of Juvenile Justice": "Corrections & Justice",
    "Department of State Police": "Public Safety & Emergency",
    "Department of Military Affairs": "Public Safety & Emergency",
    "Illinois Emergency Management Agency": "Public Safety & Emergency",
    "Department of Human Services": "Health & Human Services",
    "Department of Public Health": "Health & Human Services",
    "Department of Veterans' Affairs": "Health & Human Services",
    "Illinois Medical District Commission": "Health & Human Services",
    "Department of Natural Resources": "Natural Resources & Environment",
    "Department of Agriculture": "Natural Resources & Environment",
    "Department of Transportation": "Infrastructure & Operations",
    "Department of Central Management Services": "Infrastructure & Operations",
    "Illinois Courts": "Legal & Judicial",
    "Appellate Court / Second District": "Legal & Judicial",
    "Appellate Court / Third District": "Legal & Judicial",
    "Appellate Court / Fourth District": "Legal & Judicial",
    "Appellate Court / Fifth District": "Legal & Judicial",
    "Office of the Attorney General": "Legal & Judicial",
    "Governor's Office": "Executive & Administrative",
    "Office of the Secretary of State": "Executive & Administrative",
    "Department of Revenue": "Executive & Administrative",
    "Historic Preservation Agency": "Executive & Administrative"
}

buildingDatasetRaw["Agency Category"] = buildingDatasetRaw["Agency Name"].map(agencyCategories)
buildingDatasetRaw

In [ ]:
buildingDatasetRaw["Year Constructed"].unique()

In [ ]:
buildingDatasetCleaned = buildingDatasetRaw[(buildingDatasetRaw["Year Constructed"] > 0) & (buildingDatasetRaw["Year Acquired"] > 0) 
                                            & (buildingDatasetRaw["Square Footage"] > 0)]
buildingDatasetCleaned

In [ ]:
buildingDatasetCleaned["Year Constructed"].min(), buildingDatasetCleaned["Year Constructed"].max()

In [ ]:
buildingDatasetCleaned["Usage Description"].unique()

In [ ]:
brushSelection = alt.selection_interval(encodings = ["x", "y"])
clickSelection = alt.selection_point(fields = ["Usage Description"])

In [ ]:
buildingYearAndAreaScatterPlot = alt.Chart(buildingDatasetCleaned).mark_circle(size = 40, opacity = 0.6).encode(
    x = alt.X("Year Constructed:Q", title = "The Year of the Building Being Constructed", scale = alt.Scale(domain = [1700, 2025])),
    y = alt.Y("Square Footage:Q", title = "The Area of the Building in Square Footage ", scale = alt.Scale(type = "log")),
    color = alt.Color("Agency Category:N", title = "The Agency Category", scale = alt.Scale(scheme = "category10")),
    tooltip = [
        alt.Tooltip("Location Name:N", title = "Location Name"),
        alt.Tooltip("Agency Name:N", title = "Agency Name"),
        alt.Tooltip("Agency Category:N", title = "Agency Category"),
        alt.Tooltip("City:N", title = "City Name"),
        alt.Tooltip("Year Constructed:Q", title = "The Year Being Built"),
        alt.Tooltip("Square Footage:Q", title = "Areas in Square Footage", format = ","),
        alt.Tooltip("Total Floors:Q", title = "Total Floors"),
        alt.Tooltip("Usage Description:N", title = "Primary Use"),
        alt.Tooltip("Bldg Status:N", title = "Building Status")
        ]
).properties(
    width = 400,
    height = 400,
    title = alt.Title(
        text = "The Distribution of Buildings Based on the Year Constructed and Square Footage",
        anchor = "middle"
    )
)

buildingYearAndAreaScatterPlot

In [ ]:
agencyCategoryHistogramPlot = alt.Chart(buildingDatasetCleaned).mark_bar().encode(
    x = alt.X("Agency Category:N", title = "The Agency Category", axis = alt.Axis(labelAngle = -45)),
    y = alt.Y("count():Q", title = "The Number of Buildings"),
    color = alt.Color("Agency Category:N", title = "The Agency Category of the Building", scale = alt.Scale(scheme = "category10"), legend = None),
    tooltip = [
        alt.Tooltip("Agency Category:N", title = "The Agency Category"),
        alt.Tooltip("count():Q", title = "The Number of Buildings", format = ",")
    ]
).properties(
    width = 400,
    height = 400,
    title = alt.Title(
        text = "The Number of Buildings by Category",
        anchor = "middle"
    )
)

agencyCategoryHistogramPlot

In [ ]:
buildingAgeAndSizeAnalysis = (
    buildingYearAndAreaScatterPlot.add_params(brushSelection).properties(width = 300) | 
    agencyCategoryHistogramPlot.transform_filter(brushSelection).properties(width = 300)
).properties(
    title = {
        "text": "The Comprehensive Analysis of Illinois State Buildings Based on the Year Constructed and Square Footage by Category Distribution",
        "fontSize": 16,
        "anchor": "middle"
    }
)

buildingAgeAndSizeAnalysis

In [ ]:
buildingAgeAndSizeAnalysis.save("../assets/json/buildingAgeAndSizeAnalysis.json")

In [ ]:
usageTypeBarChart = alt.Chart(buildingDatasetCleaned).mark_bar().encode(
    x = alt.X("count():Q", title = "Number of Buildings"),
    y = alt.Y("Usage Description:N", title = "Usage Type", sort = "-x"),
    color = alt.condition(
        clickSelection, 
        alt.Color("Usage Description:N", scale = alt.Scale(scheme = "category20"), legend = None), 
        alt.value("lightgray")
    ),
    opacity = alt.condition(clickSelection, alt.value(1.0), alt.value(0.3)),
    tooltip = [
        alt.Tooltip("Usage Description:N", title = "Usage Type"), 
        alt.Tooltip("count():Q", title = "Number of Buildings", format = ",")
    ]
).properties(
    width = 300,
    height = 400,
    title = "Building Usage Types Distribution"
).add_params(clickSelection)

usageTypeBarChart

In [ ]:
topCounties = buildingDatasetCleaned["County"].value_counts().head(20).index.tolist()
topCountyBuildings = buildingDatasetCleaned[buildingDatasetCleaned["County"].isin(topCounties)]

countyDistributionBarChart = alt.Chart(topCountyBuildings).mark_bar().encode(
    x = alt.X("count():Q", title = "Number of Buildings"),
    y = alt.Y("County:N", title = "County (Top 20)", sort = "-x"),
    color = alt.value("steelblue"),
    tooltip = [
        alt.Tooltip("County:N", title = "County"),
        alt.Tooltip("count():Q", title = "Number of Buildings", format = ",")
    ]
).properties(
    width = 350,
    height = 400,
    title = "The Distribution of Buildings by County (Top 20 Counties)"
)

countyDistributionBarChart

In [ ]:
buildingUsageTypeAndLocationAnalysis = (
    usageTypeBarChart | 
    countyDistributionBarChart.transform_filter(clickSelection)
).properties(
    title = {
        "text": "The Comprehensive Analysis of Illinois State Buildings Based on the Building Usage Types and Location by County Distribution",
        "fontSize": 16,
        "anchor": "middle"
    }
)

buildingUsageTypeAndLocationAnalysis

In [ ]:
buildingUsageTypeAndLocationAnalysis.save('../assets/json/buildingUsageTypeAndLocationAnalysis.json')